# 04 — Cite evidence and abstain with a policy

**Track:** Beginner · **Stage:** Generation & Guardrails

An answer can contain a link and still be unsupported. A trustworthy system preserves evidence identity through retrieval, context construction, generation, validation, and rendering. In this notebook, you will build a strict prompt that forces the LLM to cite its sources, and a basic output parser that detects when the model abstains.

## What you will build

- A LangChain prompt designed for strict citations.
- An abstention guardrail to prevent hallucination when retrieval fails to find the answer.

## Setup

We use LangChain to construct our pipeline. For deterministic learning without API keys, we again use a mock LLM.

In [ ]:
# !pip install langchain langchain-core

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms.fake import FakeListLLM

## 1. The Corpus

Assume retrieval has already happened. We have two documents in our context window.

In [ ]:
retrieved_docs = [
    Document(
        page_content="Enterprise customers receive a status update within 30 minutes of a confirmed P1 incident.",
        metadata={"source": "sla_policy.md", "id": "chunk-01"}
    ),
    Document(
        page_content="To page the on-call engineer, use the /page command in the #incidents Slack channel.",
        metadata={"source": "incident_response.md", "id": "chunk-02"}
    )
]

def format_docs_with_citations(docs):
    # We inject the source ID directly into the context window so the LLM can reference it.
    return "\n\n".join(f"[Source: {d.metadata['source']}] {d.page_content}" for d in docs)

## 2. Prompt Engineering for Citations and Abstention

We instruct the model to do two things:
1. Only answer if the context contains the answer.
2. If it does not contain the answer, output exactly: `INSUFFICIENT_EVIDENCE`.

In [ ]:
template = """
You are an expert technical support assistant. Your job is to answer questions using ONLY the provided context.

RULES:
1. If the context does not contain the information needed to answer the question, you must reply with exactly: INSUFFICIENT_EVIDENCE
2. If you can answer the question, you must cite the source in brackets at the end of the sentence, like this: [Source: filename.md]

Context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

## 3. Creating the Guardrail

A simple output parser can intercept the `INSUFFICIENT_EVIDENCE` string and throw an error or trigger a fallback mechanism (like falling back to web search, which we will explore in the Advanced curriculum).

In [ ]:
class AbstentionGuardrail(StrOutputParser):
    def parse(self, text: str) -> str:
        cleaned = text.strip()
        if "INSUFFICIENT_EVIDENCE" in cleaned:
            return "System Abstention: The retrieved context does not contain the answer. Please escalate to a human."
        return cleaned

# Scenario 1: The model answers correctly with a citation
llm_success = FakeListLLM(responses=["Enterprise customers must receive a status update within 30 minutes [Source: sla_policy.md]."])

# Scenario 2: The model correctly abstains
llm_abstain = FakeListLLM(responses=["INSUFFICIENT_EVIDENCE"])

def build_chain(llm):
    return (
        prompt
        | llm
        | AbstentionGuardrail()
    )

context_str = format_docs_with_citations(retrieved_docs)

## 4. Testing the Guardrail

In [ ]:
print("--- Test 1: Supported Question ---")
chain_success = build_chain(llm_success)
result_1 = chain_success.invoke({"context": context_str, "question": "How fast do we update enterprise customers?"})
print(result_1)

print("\n--- Test 2: Unsupported Question ---")
chain_abstain = build_chain(llm_abstain)
result_2 = chain_abstain.invoke({"context": context_str, "question": "What is the policy for standard tier customers?"})
print(result_2)

## Reflection

1. **String Matching vs Structured Output:** We used a string (`INSUFFICIENT_EVIDENCE`) to trigger the guardrail. In production, using a model that supports structured outputs (e.g. OpenAI Tool Calling or PydanticOutputParser) returning an `is_answerable` boolean is far more robust.
2. **Safety:** Why is an explicit abstention path better than a prompt that says "try your best to answer"? Because in enterprise RAG, a hallucinated answer is always worse than no answer.